In [3]:
import numpy as np, tensorflow as tf, shutil, json
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
GRU, Dense, Dropout, BatchNormalization,
Conv1D, MaxPooling1D, GlobalAveragePooling1D)
from tensorflow.keras.callbacks import (
EarlyStopping, ModelCheckpoint, ReduceLROnPlateau)
X_tr = np.load("../data/X_train.npy")
X_val = np.load("../data/X_val.npy")
X_test = np.load("../data/X_test.npy")
y_tr = np.load("../data/y_train.npy")
y_val = np.load("../data/y_val.npy")
y_test = np.load("../data/y_test.npy")
NUM_CLASSES = y_tr.shape[1] # 383
SEQ_LEN = X_tr.shape[1] # 30
FEATURES = X_tr.shape[2] # 132
# Reuse LSTM result from Week 2
lstm_model = tf.keras.models.load_model("ml/saved_models/lstm_best.keras")
_, lstm_acc = lstm_model.evaluate(X_test, y_test, verbose=0)
print(f"LSTM accuracy: {lstm_acc*100:.2f}%")

LSTM accuracy: 32.86%


Shared callback helper

In [5]:
def get_callbacks(model_name):
    return [
        EarlyStopping(monitor="val_loss", patience=10,
                      restore_best_weights=True),
        ModelCheckpoint(
            f"ml/saved_models/{model_name}_best.keras",
            monitor="val_accuracy", save_best_only=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5),
    ]

 Train GRU model

In [6]:
gru_model = Sequential([
GRU(128, return_sequences=True, input_shape=(SEQ_LEN, FEATURES)),
BatchNormalization(), Dropout(0.3),
GRU(64, return_sequences=False),
BatchNormalization(), Dropout(0.3),
Dense(128, activation="relu"), Dropout(0.2),
Dense(NUM_CLASSES, activation="softmax")
])
gru_model.compile(optimizer="adam",
loss="categorical_crossentropy", metrics=["accuracy"])
gru_model.fit(X_tr, y_tr, epochs=100, batch_size=32,
validation_data=(X_val, y_val),
callbacks=get_callbacks("gru"), verbose=1)
_, gru_acc = gru_model.evaluate(X_test, y_test, verbose=0)
print(f"GRU Test Accuracy: {gru_acc*100:.2f}%")

Epoch 1/100


/opt/homebrew/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


93/93 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.0300 - loss: 5.5234 - val_accuracy: 0.0236 - val_loss: 5.5660 - learning_rate: 0.0010
Epoch 2/100
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.0590 - loss: 4.8369 - val_accuracy: 0.0346 - val_loss: 5.2103 - learning_rate: 0.0010
Epoch 3/100
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.0813 - loss: 4.5224 - val_accuracy: 0.0283 - val_loss: 5.8843 - learning_rate: 0.0010
Epoch 4/100
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.0971 - loss: 4.2869 - val_accuracy: 0.0535 - val_loss: 4.9775 - learning_rate: 0.0010
Epoch 5/100
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.1106 - loss: 4.0766 - val_accuracy: 0.1024 - val_loss: 4.8469 - learning_rate: 0.0010
Epoch 6/100
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.1305 - loss: 3.8514 - val_accuracy: 0.0614 - val_loss: 5.3677 - learning_rate: 0.0010
Epoch 7/100
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.1548 - loss: 3.6563 - val_accuracy

Train 1D CNN model

In [7]:
cnn_model = Sequential([
Conv1D(64, kernel_size=3, activation="relu",
input_shape=(SEQ_LEN, FEATURES)),
BatchNormalization(),
MaxPooling1D(pool_size=2),
Dropout(0.2),
Conv1D(128, kernel_size=3, activation="relu"),
BatchNormalization(),
MaxPooling1D(pool_size=2),
Dropout(0.2),
GlobalAveragePooling1D(),
Dense(128, activation="relu"), Dropout(0.3),
Dense(NUM_CLASSES, activation="softmax")
])
cnn_model.compile(optimizer="adam",
loss="categorical_crossentropy", metrics=["accuracy"])
cnn_model.fit(X_tr, y_tr, epochs=100, batch_size=32,
validation_data=(X_val, y_val),
callbacks=get_callbacks("cnn"), verbose=1)
_, cnn_acc = cnn_model.evaluate(X_test, y_test, verbose=0)
print(f"CNN Test Accuracy: {cnn_acc*100:.2f}%")

Epoch 1/100


/opt/homebrew/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.0358 - loss: 5.5225 - val_accuracy: 0.0173 - val_loss: 5.6040 - learning_rate: 0.0010
Epoch 2/100
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0637 - loss: 4.8140 - val_accuracy: 0.0394 - val_loss: 5.1282 - learning_rate: 0.0010
Epoch 3/100
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0887 - loss: 4.4534 - val_accuracy: 0.0189 - val_loss: 5.3061 - learning_rate: 0.0010
Epoch 4/100
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1133 - loss: 4.1917 - val_accuracy: 0.0598 - val_loss: 6.9727 - learning_rate: 0.0010
Epoch 5/100
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1342 - loss: 3.9533 - val_accuracy: 0.0740 - val_loss: 5.4885 - learning_rate: 0.0010
Epoch 6/100
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1524 - loss: 3.7775 - val_accuracy: 0.0472 - val_loss: 6.0875 - learning_rate: 0.0010
Epoch 7/100
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1639 - loss: 3.6022 - val_accuracy: 0.044

Compare all three and save the winner

In [ ]:
results = {"lstm": lstm_acc, "gru": gru_acc, "cnn": cnn_acc}
print("\n=== Model Comparison ===")
for name, acc in sorted(results.items(), key=lambda x: -x[1]):
    print(f"{name.upper():6s} Accuracy: {acc*100:.2f}%")
best_name = max(results, key=results.get)
print(f"\nWinner: {best_name.upper()}")
# Copy the winning model as the production model
best_path = f"ml/saved_models/{best_name}_best.keras"
shutil.copy(best_path, "ml/saved_models/mudralearn_model.keras")
print("Production model: ml/saved_models/mudralearn_model.keras")
print("Label map at : ../saved_models/label_map.json")


=== Model Comparison ===
GRU    Accuracy: 33.18%
LSTM   Accuracy: 32.86%
CNN    Accuracy: 23.27%

Winner: GRU
Production model: ml/saved_models/mudralearn_model.keras
Label map at : ml/saved_models/label_map.json


which signs the model struggles with

In [14]:
from sklearn.metrics import classification_report
with open("../saved_models/label_map.json") as f:
    label_map = json.load(f)
class_names = [label_map[str(i)] for i in range(NUM_CLASSES)]

prod = tf.keras.models.load_model("ml/saved_models/mudralearn_model.keras")
y_pred = prod.predict(X_test)
y_true = np.argmax(y_test, axis=1)
y_pred_cls = np.argmax(y_pred, axis=1)

# Show per-sign results for the first 20 signs
print(classification_report(
    y_true, y_pred_cls, labels=range(20), target_names=class_names[:20]))

20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
               precision    recall  f1-score   support

       1. one       0.00      0.00      0.00         5
      10. ten       0.00      0.00      0.00         1
   11. eleven       0.00      0.00      0.00         0
   12. twelve       0.00      0.00      0.00         0
 13. thirteen       0.00      0.00      0.00         0
 14. fourteen       0.00      0.00      0.00         1
  15. fifteen       0.00      0.00      0.00         1
  16. sixteen       0.00      0.00      0.00         0
17. seventeen       0.00      0.00      0.00         1
 18. eighteen       0.00      0.00      0.00         0
 19. nineteen       0.00      0.00      0.00         0
       2. two       0.10      0.20      0.13         5
   20. twenty       0.00      0.00      0.00         1
     3. three       0.00      0.00      0.00         0
      4. four       0.00      0.00      0.00         5
      5. five       0.00      0.00      0.00         3
       6. six       0.00 

/opt/homebrew/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/lib/python3.11/